# Multi-Day A-S Baseline Across Available DOGEUSDT Replay Days

In [1]:
import sys
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "procs").exists()), pathlib.Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from procs.gym.calibration import tune_gamma
from procs.gym.data_loader import load_multi_day
from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.helpers.fast_rollout import fast_simulate

## 1. Discover Available Replay Days

In [2]:
cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)
print(f"Running A-S baseline across {len(dates)} available DOGEUSDT replay days.")

  2025-01-01:  713,815 snapshots, σ=0.000021
  2025-01-02:  766,464 snapshots, σ=0.000035
  2025-01-03:  776,383 snapshots, σ=0.000047
  2025-01-04:  778,293 snapshots, σ=0.000045
  2025-01-05:  723,494 snapshots, σ=0.000031
  2025-01-06:  766,311 snapshots, σ=0.000035
  2025-01-07:  787,093 snapshots, σ=0.000062
  2025-01-08:  821,589 snapshots, σ=0.000052
  2025-01-09:  809,421 snapshots, σ=0.000046
  2025-01-10:  789,320 snapshots, σ=0.000045
  2025-01-11:  724,826 snapshots, σ=0.000023
  2025-01-12:  719,550 snapshots, σ=0.000022
  2025-01-13:  819,981 snapshots, σ=0.000046
  2025-01-14:  775,454 snapshots, σ=0.000038
  2025-01-15:  782,299 snapshots, σ=0.000047
  2025-01-16:  788,946 snapshots, σ=0.000048
  2025-01-17:  801,259 snapshots, σ=0.000061
  2025-01-18:  829,125 snapshots, σ=0.000074
  2025-01-19:  842,892 snapshots, σ=0.000096
  2025-01-20:  857,508 snapshots, σ=0.000115
  2025-01-21:  848,107 snapshots, σ=0.000112
  2025-01-22:  812,180 snapshots, σ=0.000044
  2025-01-

## 2. Lightweight Per-Day Calibration

In [3]:
def calibrate_from_arrays(
    S: np.ndarray,
    dt: np.ndarray,
    tick_size: float = 0.00001,
    n_depth_ticks: int = 5,
    min_arrivals: int = 10,
) -> tuple[float, float, float]:
    T = float(dt.sum())
    N = len(S)

    dS = np.diff(S)
    dt_mid = dt[1:]
    window_size = max(1, int(600.0 / np.median(dt_mid[dt_mid > 0])))
    sigma_estimates = []
    for start in range(0, N - 1 - window_size, window_size):
        dS_w = dS[start:start + window_size]
        dt_w = dt_mid[start:start + window_size]
        total_t = dt_w.sum()
        if total_t > 0:
            sigma_estimates.append(np.sqrt(np.sum(dS_w ** 2) / total_t))
    sigma = float(np.median(sigma_estimates)) if sigma_estimates else float(np.sqrt(np.sum(dS ** 2) / T))

    mid_diff = np.abs(np.diff(S))
    arrival_mask = mid_diff >= tick_size * 0.5
    arrival_depths = mid_diff[arrival_mask]

    bin_edges = np.arange(0, n_depth_ticks + 1) * tick_size
    bin_centres = (bin_edges[:-1] + bin_edges[1:]) / 2
    counts, _ = np.histogram(arrival_depths, bins=bin_edges)
    lambda_emp = counts / T

    valid = counts >= min_arrivals
    if valid.sum() < 2:
        return sigma, float(len(arrival_depths) / T), 35_000.0

    x = bin_centres[valid]
    y = np.log(lambda_emp[valid])
    n = len(x)
    sx = x.sum(); sy = y.sum()
    sx2 = (x ** 2).sum(); sxy = (x * y).sum()
    slope = (n * sxy - sx * sy) / (n * sx2 - sx ** 2)
    intercept = (sy - slope * sx) / n

    kappa = float(-slope) if np.isfinite(slope) and slope < 0 else 35_000.0
    A = float(np.exp(intercept)) if np.isfinite(intercept) else 0.8
    return sigma, A, kappa

## 3. Run the Per-Day Baseline

In [ ]:
results = []

for i, (S, dt, date) in enumerate(zip(daily_S, daily_dt, dates), start=1):
    T = float(dt.sum())
    sigma, A, kappa = calibrate_from_arrays(S, dt, tick_size=cfg.tick_size)
    as_gamma, _ = tune_gamma(
        midprices=S,
        dt_array=dt,
        sigma=sigma,
        kappa=kappa,
        A=A,
        tick_size=cfg.tick_size,
        Q_MAX=cfg.q_max,
        gamma_range=cfg.as_gamma_range,
        n_trials=cfg.as_gamma_trials,
        num_trajectories=cfg.evaluation_rollouts,
        seed=cfg.evaluation_seed,
        verbose=False,
    )
    stats = fast_simulate(
        midprices=S,
        dt_array=dt,
        gamma=as_gamma,
        sigma=sigma,
        kappa=kappa,
        A=A,
        terminal_time=T,
        tick_size=cfg.tick_size,
        Q_MAX=cfg.q_max,
        num_trajectories=cfg.evaluation_rollouts,
        seed=cfg.evaluation_seed,
        use_linear_approximation=False,
    )
    results.append({
        "Day": date,
        "Sharpe": float(stats["sharpe"].mean()),
        "Sortino": float(stats["sortino"].mean()),
        "Max DD": float(stats["max_drawdown"].mean()),
        "P&L-to-MAP": float(stats["pnl_to_map"].mean()),
        "Final PnL": float(stats["total_pnl"].mean()),
        "Mean |q|": float(stats["mean_abs_q"].mean()),
        "Near Cap Fraction": float(stats["near_cap_fraction"].mean()),
        "sigma": sigma,
        "A": A,
        "kappa": kappa,
        "as_gamma": as_gamma,
    })
    print(
        f"  [{i:2d}/{len(dates):2d}] {date}  gamma={as_gamma:.4f}  sigma={sigma:.6f}  "
        f"kappa={kappa:.0f}  A={A:.3f}  Sharpe={stats['sharpe'].mean():+.4f}  "
        f"MaxDD={stats['max_drawdown'].mean():.4f}  PnL={stats['total_pnl'].mean():+.4f}"
    )

df = pd.DataFrame(results).set_index("Day")
print(f"\nDone: {len(df)} available days.")


  [ 1/29] 2025-01-01  gamma=0.9487  sigma=0.000020  kappa=39822  A=0.256  Sharpe=+0.0452  MaxDD=0.0048  PnL=+0.3406


## 4. Summaries and Plots

In [ ]:
summary = pd.DataFrame({
    col: [df[col].mean(), df[col].std(ddof=0), df[col].median()]
    for col in df.columns
}, index=["Mean", "Std", "Median"])

full = pd.concat([df, summary])
print(full.to_string(float_format="%.6f"))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL", "sigma"]
colors = ["steelblue", "seagreen", "indianred", "mediumpurple", "darkorange", "grey"]

for ax, metric, color in zip(axes.flat, metrics, colors):
    ax.bar(range(len(df)), df[metric], color=color, alpha=0.8)
    ax.axhline(y=df[metric].mean(), color="k", ls="--", lw=1, label=f"mean={df[metric].mean():.4f}")
    ax.set_title(metric)
    ax.set_xlabel("Day")
    ax.set_ylabel(metric)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df.index, rotation=45, ha="right", fontsize=7)
    ax.legend(fontsize=8)

plt.suptitle(f"A-S Baseline (per-day calibration) - {len(df)} Available Days {cfg.pair}", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
output_path = cfg.result_path(f"as_baseline_{len(df)}day_results.csv")
df.to_csv(output_path)
print(f"Saved detailed per-day results to {output_path}")